In [1]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# Set file paths
data_dir = "./1-clustering"  # Directory containing clustering results
output_dir = "./1-train_test_split"  # Directory to save train-test split results
os.makedirs(output_dir, exist_ok=True)

# File list
files = [
    "AM-I-filtered/AM-I-filtered_with_labels_k4.csv",
    'AM-II-filtered/AM-II-filtered_with_labels_k4.csv'
]

# Random seed
random_seed = 42

# Main process
for file in files:
    file_path = os.path.join(data_dir, file)
    df = pd.read_csv(file_path)  # Read clustering result file

    # Remove rows with missing values
    df = df.dropna()

    # Initialize empty DataFrames to store train and test data for the current file
    train_data = pd.DataFrame()
    test_data = pd.DataFrame()

    # Group by UMAP_Cluster
    for cluster in df['UMAP_Cluster'].unique():
        cluster_df = df[df['UMAP_Cluster'] == cluster].reset_index(drop=True)

        # Split into train and test sets
        train_df, test_df = train_test_split(cluster_df, test_size=0.1, random_state=random_seed)

        # Append current cluster's train and test data to the main DataFrames
        train_data = pd.concat([train_data, train_df], ignore_index=True)
        test_data = pd.concat([test_data, test_df], ignore_index=True)

    # Check for duplicate samples between train and test sets
    duplicate_rows = train_data.merge(test_data, how='inner')
    if not duplicate_rows.empty:
        print(f"⚠️ Potential data leakage detected (duplicate samples in train and test sets): {file}")
        print(f"Number of duplicate samples = {len(duplicate_rows)}")
    else:
        print(f"✅ No duplicate samples between train and test sets: {file}")

    # Create output filenames (remove directory path, keep only filename)
    base_filename = os.path.basename(file_path)  # Get filename, e.g., "Default-2_with_labels_k4.csv"
    base_name_without_ext = os.path.splitext(base_filename)[0]  # Remove extension
    
    # Save train and test sets for the current file
    train_output_path = os.path.join(output_dir, f"{base_name_without_ext}_train.csv")
    test_output_path = os.path.join(output_dir, f"{base_name_without_ext}_test.csv")

    train_data.to_csv(train_output_path, index=False)
    test_data.to_csv(test_output_path, index=False)

    print(f"Saved train data for {file} to {train_output_path}")
    print(f"Saved test data for {file} to {test_output_path}")

✅ No duplicate samples between train and test sets: AM-I-filtered/AM-I-filtered_with_labels_k4.csv
Saved train data for AM-I-filtered/AM-I-filtered_with_labels_k4.csv to ./1-train_test_split/AM-I-filtered_with_labels_k4_train.csv
Saved test data for AM-I-filtered/AM-I-filtered_with_labels_k4.csv to ./1-train_test_split/AM-I-filtered_with_labels_k4_test.csv
✅ No duplicate samples between train and test sets: AM-II-filtered/AM-II-filtered_with_labels_k4.csv
Saved train data for AM-II-filtered/AM-II-filtered_with_labels_k4.csv to ./1-train_test_split/AM-II-filtered_with_labels_k4_train.csv
Saved test data for AM-II-filtered/AM-II-filtered_with_labels_k4.csv to ./1-train_test_split/AM-II-filtered_with_labels_k4_test.csv
